In [1]:
import contextlib, io, os, json, zipfile
from getpass import getpass
from pathlib import Path
import numpy as np
import pandas as pd
import finlab
from finlab import data
from IPython.display import display

finlab.login()  # 憑證已快取時不會再開瀏覽器

已登入（使用快取憑證）。


In [2]:
STOCK_IDS: list[str] = [
    # "1303", "4958", "6505", "8046", "3189", "4938", "1301", "3533", "1802", "6415",
    # "1326", "2603", "2368", "2891", "3034", "3037", "2409", "3044", "3653", "2379",
    # "2383", "2881", "3231", "2609", "2313", "6446", "2882", "2357", "3443", "3711",
    "2317", "3008", "2345", "2404", "2330", "2887", "3661", "2376", "2382", "2883",
]

# 設定全域股票範圍（whitelist 限制在這 40 檔）
# 之後所有 data.get() 與 data.indicator() 只會下載並回傳這 40 檔的資料
data.set_universe(whitelist=STOCK_IDS)

In [19]:
START_DATE = "2026-09-01"

1. 最新季 ROE (基本面)

In [10]:
def roe_latest():
    # 取得全體股票季ROE資料（不篩選特定個股、不排序）
    roe = data.get("fundamental_features:ROE稅後")

    # 取得指定季度的資料列，保留股票代號欄位標題與該列數據，並保留四捨五入
    roe_q = roe.loc[["2026-Q2"]].round(2)

    return roe_q

roe_latest()

symbol   1301  1303  1326  1802  2313  2317   2330   2345  2357   2368  ...  \
date                                                                    ...   
2026-Q2  2.21  6.13  1.34  2.43   3.1  3.41  11.39  16.42  6.29  12.41  ...   

symbol   3533  3653  3661  3711  4938  4958  6415  6446  6505  8046  
date                                                                 
2026-Q2  4.53  10.3  3.16  5.37  2.51  1.93  3.94   7.1  4.76  4.71  

[1 rows x 40 columns]

2. 最新月營收年增率 (基本面)

In [11]:
def revenue_yoy_latest():

    # 取得個股月營收年增率
    revenue = data.get("monthly_revenue:去年同月增減(%)", start=START_DATE)

    # 最新月份年增率，單位：%
    revenue_yoy_latest_pct = revenue.iloc[[-1]].round(2)

    return revenue_yoy_latest_pct
revenue_yoy_latest()

symbol      1301   1303   1326  1802   2313   2317   2330   2345   2357  \
date                                                                      
2026-10-12  7.13  55.25  44.77  34.1  15.01  38.41  54.64  66.12  17.73   

symbol       2368  ...   3533   3653    3661   3711   4938   4958  6415  \
date               ...                                                    
2026-10-12  55.46  ...  16.12  79.96  495.99  45.52  86.87  36.12  50.7   

symbol       6446   6505   8046  
date                             
2026-10-12  99.97  65.59  57.81  

[1 rows x 40 columns]

3. 最新月營收月增率 (基本面)

In [12]:
def revenue_mom_latest(stock_id="2330"):

    # 取得個股月營收月增率
    revenue = data.get("monthly_revenue:上月比較增減(%)", start=START_DATE)

    # 最新月份月增率，單位：%
    revenue_mom_latest_pct = revenue.iloc[[-1]].round(2)

    return revenue_mom_latest_pct
revenue_mom_latest()

symbol      1301   1303  1326  1802   2313   2317  2330  2345  2357   2368  \
date                                                                         
2026-10-12  7.33  12.49  9.29  8.18  22.89  25.69 -0.57 -1.15   2.4 -10.24   

symbol      ...  3533  3653  3661  3711    4938   4958  6415  6446  6505  8046  
date        ...                                                                 
2026-10-12  ...  4.95 -5.32  13.7  7.15  143.19  27.54   3.9  0.69  6.13  4.35  

[1 rows x 40 columns]

4. 收盤價 (技術面)

In [13]:
def close_latest():

    # 取得個股未還原收盤價
    close = data.get("price:收盤價", start=START_DATE)

    # 最新收盤價，單位：元／股
    close_twd = close.iloc[[-1]].round(2)

    return close_twd
close_latest()

symbol      1301   1303  1326  1802   2313   2317    2330    2345   2357  \
date                                                                       
2026-10-22  73.8  301.0  73.0  67.1  224.5  245.0  2385.0  1775.0  946.0   

symbol        2368  ...    3533    3653    3661   3711   4938   4958   6415  \
date                ...                                                       
2026-10-22  1115.0  ...  1795.0  6070.0  3495.0  654.0  103.5  567.0  472.0   

symbol        6446   6505    8046  
date                               
2026-10-22  1160.0  106.5  1265.0  

[1 rows x 40 columns]

5. 近5日報酬 (技術面)

In [14]:
def return_5d():

    # 取得個股還原收盤價
    p = data.get("etl:adj_close", start=START_DATE)

    # 近5日報酬，單位：%
    return_5d_pct = ((p.iloc[[-1]] / p.iloc[[-6]] - 1) * 100).round(2)

    return return_5d_pct
return_5d()

symbol      1301  1303  1326  1802  2313  2317  2330  2345  2357  2368  ...  \
date                                                                    ...   
2026-10-22 -9.67 -4.29 -5.19  7.02 -1.54   2.3 -2.25 -8.97 -2.97  -6.3  ...   

symbol      3533   3653  3661  3711  4938  4958  6415  6446  6505  8046  
date                                                                     
2026-10-22  -1.1 -10.27 -5.67 -2.82   5.5  9.25  1.18 -3.73 -4.48 -5.95  

[1 rows x 40 columns]

6. 近20日報酬 (技術面)

In [15]:
def return_20d():

    # 取得個股還原收盤價
    p = data.get("etl:adj_close", start=START_DATE)

    # 近20日報酬，單位：%
    return_20d_pct = ((p.iloc[[-1]] / p.iloc[[-21]] - 1) * 100).round(2)

    return return_20d_pct
return_20d()

symbol       1301   1303  1326   1802  2313  2317  2330  2345  2357  2368  \
date                                                                        
2026-10-22  13.19  25.68  7.67  11.65  1.13  -2.0 -3.83 -3.53   0.0   6.7   

symbol      ...   3533  3653  3661  3711   4938   4958   6415  6446   6505  \
date        ...                                                              
2026-10-22  ...  12.89  4.84 -4.64 -1.36  13.99  20.64  10.41  0.87  20.34   

symbol       8046  
date               
2026-10-22  19.34  

[1 rows x 40 columns]

7. 個股跟大盤近20相對報酬 (技術面)

In [16]:
import pandas as pd


def excess_return_20d():
    # 取得個股還原收盤價與大盤指數
    p = data.get("etl:adj_close", start=START_DATE)
    b = data.get("benchmark_return:發行量加權股價報酬指數", start=START_DATE)

    # 以收盤價 p 的 index 為基準對齊大盤，收盤價沒有的日期直接忽略
    b = b.squeeze("columns").reindex(p.index).ffill()

    # 計算最後一筆相較於前 20 個交易日的報酬率 (%)
    # 加上 .values 避免日期 index 不同而對齊出 NaN
    stock_return_20d = (p.iloc[[-1]] / p.iloc[-21].values - 1) * 100
    bench_return_20d = (b.iloc[-1] / b.iloc[-21] - 1) * 100

    # 個股報酬 - 大盤報酬
    excess_return_20d_pp = (stock_return_20d - bench_return_20d).round(2)

    return excess_return_20d_pp

excess_return_20d()

symbol      1301   1303   1326   1802  2313  2317  2330  2345  2357   2368  \
date                                                                         
2026-10-22  16.7  29.19  11.18  15.15  4.63  1.51 -0.32 -0.03  3.51  10.21   

symbol      ...  3533  3653  3661  3711   4938   4958   6415  6446   6505  \
date        ...                                                             
2026-10-22  ...  16.4  8.34 -1.13  2.15  17.49  24.15  13.92  4.38  23.85   

symbol       8046  
date               
2026-10-22  22.85  

[1 rows x 40 columns]

8. 5日均量／20日均量 (技術面)

In [17]:
def volume_ma5_to_ma20():

    # 取得個股成交股數
    volume = data.get("price:成交股數", start=START_DATE)

    # 計算5日與20日平均成交股數
    volume_ma5 = volume.rolling(5).mean().iloc[[-1]]
    volume_ma20 = volume.rolling(20).mean().iloc[[-1]]

    # 均量比，單位：倍
    volume_ma5_to_ma20_ratio = (volume_ma5 / volume_ma20).round(2)

    return volume_ma5_to_ma20_ratio
volume_ma5_to_ma20()

symbol      1301  1303  1326  1802  2313  2317  2330  2345  2357  2368  ...  \
date                                                                    ...   
2026-10-22   0.8  1.15  0.79  2.15  1.34  0.92   1.1  0.98  1.19  0.93  ...   

symbol      3533  3653  3661  3711  4938  4958  6415  6446  6505  8046  
date                                                                    
2026-10-22   0.6  1.08  0.57  0.96  1.09   0.9  1.77  1.07  1.17  0.75  

[1 rows x 40 columns]

9. 外資近5日買賣超強度 (籌碼面)

In [18]:
def foreign_net_buy_intensity_5d():

    # 取得個股成交股數
    volume = data.get("price:成交股數", start=START_DATE)

    # 取得外資買賣超股數
    foreign = data.get("institutional_investors_trading_summary:外陸資買賣超股數(不含外資自營商)", start=START_DATE)

    # 對齊交易日期
    foreign = foreign.reindex(volume.index)

    # 近5日外資買賣超股數／近5日成交股數，單位：%
    foreign_net_buy_intensity_5d_pct = (foreign.rolling(5).sum().iloc[[-1]] / volume.rolling(5).sum().iloc[[-1]] * 100).round(2)

    return foreign_net_buy_intensity_5d_pct
foreign_net_buy_intensity_5d()


symbol      1301  1303  1326  1802  2313   2317   2330  2345  2357  2368  ...  \
date                                                                      ...   
2026-10-22 -15.9 -5.31 -4.06  1.02 -2.52  22.88 -40.05 -5.93 -1.56 -9.49  ...   

symbol       3533  3653  3661   3711   4938  4958  6415  6446   6505   8046  
date                                                                         
2026-10-22 -12.15 -1.93 -7.41 -13.93  27.68 -3.27 -3.51 -6.02  14.86 -12.03  

[1 rows x 40 columns]

10. 外資最新交易日買賣超股數 (籌碼面)

In [19]:
def foreign_net_buy_1d(stock_id="2330"):

    # 取得外資買賣超股數
    foreign = data.get("institutional_investors_trading_summary:外陸資買賣超股數(不含外資自營商)", start=START_DATE)

    # 最新交易日買賣超，單位：股
    foreign_net_buy_1d_shares = foreign.iloc[[-1]]

    return foreign_net_buy_1d_shares
foreign_net_buy_1d()

symbol            1301        1303       1326        1802       2313  \
date                                                                   
2026-10-22 -16065504.0 -20416205.0 -8876465.0  25235718.0 -5006961.0   

symbol           2317        2330      2345      2357      2368  ...    3533  \
date                                                             ...           
2026-10-22  1739424.0 -12694626.0 -240640.0 -821452.0  368093.0  ... -8417.0   

symbol          3653      3661       3711       4938       4958      6415  \
date                                                                        
2026-10-22  311892.0 -107861.0 -5437744.0  1879147.0 -2981974.0 -389712.0   

symbol         6446        6505       8046  
date                                        
2026-10-22 -28400.0  12635429.0 -1033261.0  

[1 rows x 40 columns]

11. 投信近5日買賣超強度 (籌碼面)

In [20]:
def trust_net_buy_intensity_5d():

    # 取得個股成交股數
    volume = data.get("price:成交股數", start=START_DATE)

    # 取得投信買賣超股數
    trust = data.get("institutional_investors_trading_summary:投信買賣超股數", start=START_DATE)

    # 對齊交易日期
    trust = trust.reindex(volume.index)

    # 近5日投信買賣超股數／近5日成交股數，單位：%
    trust_net_buy_intensity_5d_pct = (trust.rolling(5).sum().iloc[[-1]] / volume.rolling(5).sum().iloc[[-1]] * 100).round(2)

    return trust_net_buy_intensity_5d_pct
trust_net_buy_intensity_5d()

symbol      1301  1303  1326  1802  2313  2317  2330  2345  2357   2368  ...  \
date                                                                     ...   
2026-10-22 -0.46  0.19  2.69  0.36  1.63  0.89  5.58  5.05 -5.87  10.21  ...   

symbol      3533  3653  3661  3711  4938  4958  6415  6446  6505  8046  
date                                                                    
2026-10-22  5.36  0.36  2.28  3.85 -4.82  2.26  0.29  2.71 -6.02  5.41  

[1 rows x 40 columns]

12. 投信最新交易日買賣超股數 (籌碼面)

In [21]:
def trust_net_buy_1d():

    # 取得投信買賣超股數
    trust = data.get("institutional_investors_trading_summary:投信買賣超股數", start=START_DATE)

    # 最新交易日買賣超，單位：股
    trust_net_buy_1d_shares = trust.iloc[[-1]]

    return trust_net_buy_1d_shares
trust_net_buy_1d()

symbol        1301      1303      1326       1802      2313      2317  \
date                                                                    
2026-10-22  1000.0 -475000.0  600000.0  2968000.0  141000.0 -255528.0   

symbol           2330     2345     2357      2368  ...  3533      3653  \
date                                               ...                   
2026-10-22  1349951.0  86000.0 -62400.0  497550.0  ...   0.0 -190500.0   

symbol        3661       3711      4938      4958     6415    6446       6505  \
date                                                                            
2026-10-22  1000.0  2251000.0 -740000.0  134000.0 -51000.0  1908.0 -8119000.0   

symbol          8046  
date                  
2026-10-22  289000.0  

[1 rows x 40 columns]

13. 20日平均成交金額 (技術面)

In [22]:
def amount_ma20():

    # 取得個股成交金額
    amount = data.get("price:成交金額", start=START_DATE)

    # 20日平均成交金額，保留原始金額單位
    amount_ma20_twd = (amount.rolling(20).mean().iloc[[-1]]).round(2)

    return amount_ma20_twd
amount_ma20()

symbol              1301          1303          1326          1802  \
date                                                                 
2026-10-22  6.150879e+09  2.829890e+10  3.166358e+09  5.986250e+09   

symbol              2313          2317          2330          2345  \
date                                                                 
2026-10-22  7.918830e+09  1.029193e+10  6.764174e+10  9.390124e+09   

symbol              2357          2368  ...          3533          3653  \
date                                    ...                               
2026-10-22  2.159238e+09  1.440030e+10  ...  3.303007e+09  8.679253e+09   

symbol              3661          3711          4938          4958  \
date                                                                 
2026-10-22  9.483299e+09  1.104161e+10  1.745864e+09  1.918376e+10   

symbol              6415          6446          6505          8046  
date                                                                
202

14. 10日報酬波動率 (技術面)

In [23]:
def volatility_10d():

    # 取得個股還原收盤價
    p = data.get("etl:adj_close", start=START_DATE)

    # 每日報酬
    daily_return = p / p.shift(1) - 1

    # 最近10日日報酬的樣本標準差，單位：%，未年化
    volatility_10d_daily_pct = (daily_return.rolling(10).std(ddof=1).iloc[[-1]] * 100).round(2)

    return volatility_10d_daily_pct
volatility_10d()

symbol      1301  1303  1326  1802  2313  2317  2330  2345  2357  2368  ...  \
date                                                                    ...   
2026-10-22  5.69  6.14  5.37  4.44  4.98  1.97  0.89  3.61  2.01  5.03  ...   

symbol      3533  3653  3661  3711  4938  4958  6415  6446  6505  8046  
date                                                                    
2026-10-22  3.11   3.4  3.81  3.94  3.78   4.7  4.91  2.57  6.33  3.18  

[1 rows x 40 columns]

15. 20日均線近5日變化率 (技術面)

In [24]:
def ma20_change_5d():

    # 取得個股還原收盤價
    p = data.get("etl:adj_close", start=START_DATE)

    # 計算20日均線
    ma20 = p.rolling(20).mean()

    # 最新20日均線相較5個交易日前的變化，單位：%
    ma20_change_5d_pct = ((ma20.iloc[[-1]] / ma20.iloc[[-6]] - 1) * 100).round(2)

    return ma20_change_5d_pct
ma20_change_5d()

symbol      1301  1303  1326  1802  2313  2317  2330  2345  2357  2368  ...  \
date                                                                    ...   
2026-10-22  4.76  6.88  2.88  3.54  1.98 -0.51 -0.15  0.96  0.79  4.48  ...   

symbol      3533  3653  3661  3711  4938  4958  6415  6446  6505  8046  
date                                                                    
2026-10-22  3.65   4.1  1.37  1.63  3.74  3.88  3.33 -0.11   7.0  4.71  

[1 rows x 40 columns]

16. 20日均線乖離率 (技術面)

In [25]:
def price_vs_ma20():

    # 取得個股還原收盤價
    p = data.get("etl:adj_close", start=START_DATE)

    # 最新20日均線
    ma20 = p.rolling(20).mean().iloc[[-1]]

    # 最新價格相對20日均線的乖離，單位：%
    price_vs_ma20_pct = ((p.iloc[-1] / ma20 - 1) * 100).round(2)

    return price_vs_ma20_pct
price_vs_ma20()

symbol      1301  1303  1326  1802  2313  2317  2330  2345  2357  2368  ...  \
date                                                                    ...   
2026-10-22   0.3  5.06 -0.53  7.45 -5.36  -1.3 -4.03 -9.15 -2.85 -5.41  ...   

symbol      3533   3653  3661  3711  4938  4958  6415  6446  6505  8046  
date                                                                     
2026-10-22 -1.17 -10.28 -9.51 -7.37  8.51  4.36  5.56 -1.92  2.18 -7.53  

[1 rows x 40 columns]

17. RSI14 (技術面)

In [27]:
def rsi_14():

    # 使用還原價格計算14日RSI
    rsi = data.indicator("RSI", adjust_price=True, timeperiod=14)

    # 最新RSI，範圍0～100
    rsi_14 = (rsi.iloc[[-1]]).round(2)

    return rsi_14
rsi_14()

             1301   1303   1326   1802   2313   2317   2330   2345   2357  \
date                                                                        
2026-10-22  51.61  56.57  50.32  60.55  44.08  47.35  37.91  37.42  45.09   

             2368  ...   3533   3653   3661   3711   4938   4958   6415  \
date               ...                                                    
2026-10-22  46.63  ...  50.82  42.88  39.31  43.98  67.49  56.74  55.83   

             6446   6505   8046  
date                             
2026-10-22  43.49  54.84  45.54  

[1 rows x 40 columns]

## 18. KD（5、3、3）

KD用來觀察收盤價在近期高低價區間的位置，數值通常介於0～100。本版使用TA-Lib STOCH，以還原價格計算：

- **RSV（未成熟隨機值）**＝（收盤價－近5日最低價）／（近5日最高價－近5日最低價）×100。
- **K值**＝RSV的3日簡單移動平均，反應較快。
- **D值**＝K值的3日簡單移動平均，變化較平滑。
- `matype=0`代表簡單移動平均（SMA），期間內每筆資料權重相同；與部分看盤軟體採遞迴平滑的KD可能不同。

判讀時，可觀察K、D的高低、變化方向、兩線差距與交叉情況。80以上常稱高檔、20以下常稱低檔，但高檔不一定立刻下跌，低檔也不一定立刻上漲。

### 回傳數值與時間脈絡（精簡6欄）

| 欄位 | 說明 |
|---|---|
| `k_5` | 最新K值，觀察目前高低位置 |
| `d_5` | 最新D值，搭配K判斷目前兩線關係 |
| `k_change_3d` | 最新K減3個交易日前K；正值表示上升、負值表示下降，單位為指標點數 |
| `kd_above_d_days` | K連續高於D的交易日數；相等或缺值即中斷 |
| `kd_below_d_days` | K連續低於D的交易日數；相等或缺值即中斷 |
| `kd_cross_days_ago_20d` | 最近20個交易日內，最近一次交叉距今幾個交易日，不分方向；0＝今天、1＝前一交易日、-1＝這20日未發生 |

向上交叉指前一日K≤D、最新一天K>D；向下交叉指前一日K≥D、最新一天K<D。前一日相等、當日嚴格超越也算交叉，當日剛好相等則不算。合併欄位只記錄最近一次交叉時間，不保留歷史交叉方向。

目前K在D上方時，`kd_above_d_days`大於0；在D下方時，`kd_below_d_days`大於0。兩者皆為0代表目前原始K與D相等，不推定交叉方向。若交叉距今天數為0，可用哪一個連續天數大於0辨認今日交叉方向。

連續天數依可取得歷史往回計算，不限於20日；事件距今天數只搜尋最近20個交易日。-1不表示歷史上從未交叉。

完整輸出至少需要最近21筆有效K、D資料；不足或缺漏時報錯，不跳過缺值。所有計算先使用原始精度，輸出才四捨五入至小數第二位。K、D四捨五入後可能看似相等，此時請依連續天數判斷原始兩線關係。


In [15]:
def kd_latest():
    import pandas as pd

    # KD觀察收盤價在近期高低價區間的位置，數值通常介於0～100。
    # RSV＝（收盤價－近5日最低價）／（近5日最高價－近5日最低價）×100。
    # K＝RSV的3日簡單平均；D＝K的3日簡單平均；K反應較快，D較平滑。
    # 本版使用SMA平滑，與部分看盤軟體的遞迴平滑KD可能不同。
    # 80以上常稱高檔、20以下常稱低檔，但不代表即將反轉。
    k, d = data.indicator(
        "STOCH", adjust_price=True,
        fastk_period=5,
        slowk_period=3, slowk_matype=0,
        slowd_period=3, slowd_matype=0
    )

    # 自動抓取第一檔股票的欄位名稱
    stock_id = k.columns[0]
    k = k[stock_id].sort_index()
    d = d[stock_id].reindex(k.index)

    # 最近20個交易日的交叉需21筆完整資料，不跳過缺值。
    if len(k) < 21 or pd.concat([k, d], axis=1).iloc[-21:].isna().any().any():
        raise ValueError("最近21個交易日的KD資料不足，無法完整計算近期交叉")

    # 近3日變化：最新K－3個交易日前K，單位為指標點數。
    k_change_3d = k.iloc[-1] - k.iloc[-4]

    # 從最新一天往回計算K高於／低於D的連續交易日數。
    # 使用原始精度；相等或缺值就中斷，不額外限制為20日。
    gap = k - d
    above_d_days = 0
    below_d_days = 0
    for value in gap.iloc[::-1]:
        if pd.isna(value) or value <= 0:
            break
        above_d_days += 1
    for value in gap.iloc[::-1]:
        if pd.isna(value) or value >= 0:
            break
        below_d_days += 1

    # 合併向上／向下交叉，只保留最近一次交叉距今天數。
    # 前日相等、今日嚴格超越也算交叉；今日剛好相等不算。
    cross_up = (k.shift(1) <= d.shift(1)) & (k > d)
    cross_down = (k.shift(1) >= d.shift(1)) & (k < d)
    crossed = cross_up | cross_down
    cross_days_ago = -1
    for days_ago in range(20):
        if crossed.iloc[-1 - days_ago]:
            cross_days_ago = days_ago
            break

    # 輸出才四捨五入；連續天數可辨認四捨五入後看似相等的兩線關係。
    return {
        "k_5": round(float(k.iloc[-1]), 2),  # 最新K
        "d_5": round(float(d.iloc[-1]), 2),  # 最新D
        "k_change_3d": round(float(k_change_3d), 2),  # K近3日變化
        "kd_above_d_days": above_d_days,  # K連續高於D的交易日數
        "kd_below_d_days": below_d_days,  # K連續低於D的交易日數
        "kd_cross_days_ago_20d": cross_days_ago  # 0＝今天；1＝前一交易日；-1＝近20日未交叉
    }
kd_latest()

{'k_5': 47.6,
 'd_5': 52.23,
 'k_change_3d': -12.23,
 'kd_above_d_days': 0,
 'kd_below_d_days': 5,
 'kd_cross_days_ago_20d': 4}

In [20]:
k, d = data.indicator(
    "STOCH", adjust_price=True,
    fastk_period=5,
    slowk_period=3, slowk_matype=0,
    slowd_period=3, slowd_matype=0
)

In [23]:
k.tail(5)

                 2317       2330       2345       2376       2382       2404  \
date                                                                           
2026-10-29  72.183236  60.606061  11.665813  43.780319  43.827160  44.444444   
2026-10-30  59.837557  52.929293  13.438300  34.057609  34.567901  41.239316   
2026-11-02  54.775828  31.212121  11.751792  18.581419  22.222222  24.572650   
2026-11-03  54.320988  13.333333  11.404712  16.906205  27.294686   5.415561   
2026-11-04  47.603486  13.274854  13.129858  17.454952  19.947016   8.401391   

                 2883       2887       3008       3661  
date                                                    
2026-10-29  67.128358  68.893894  28.231293  53.508772  
2026-10-30  48.104235  63.708709  27.510204  51.960784  
2026-11-02  28.948391  50.555556  10.474576  26.074731  
2026-11-03   9.669059  34.444444  10.474576   7.511598  
2026-11-04   9.124007  25.015015  14.363115  14.271744  

In [22]:
d

                 2317       2330       2345       2376       2382       2404  \
date                                                                           
2007-04-23        NaN        NaN        NaN        NaN        NaN        NaN   
2007-04-24        NaN        NaN        NaN        NaN        NaN        NaN   
2007-04-25        NaN        NaN        NaN        NaN        NaN        NaN   
2007-04-26        NaN        NaN        NaN        NaN        NaN        NaN   
2007-04-27        NaN        NaN        NaN        NaN        NaN        NaN   
...               ...        ...        ...        ...        ...        ...   
2026-10-29  76.365605  42.929293   8.779350  42.840437  61.087596  39.814815   
2026-10-30  69.974009  50.976431  11.913940  40.031937  47.724868  40.598291   
2026-11-02  62.265540  48.249158  12.285302  32.139782  33.539095  36.752137   
2026-11-03  56.311458  32.491582  12.198268  23.181744  28.028270  23.742509   
2026-11-04  52.233434  19.273436  12.095

In [14]:
def kd_latest(stock_id):
    import pandas as pd

    # KD觀察收盤價在近期高低價區間的位置，數值通常介於0～100。
    # RSV＝（收盤價－近5日最低價）／（近5日最高價－近5日最低價）×100。
    # K＝RSV的3日簡單平均；D＝K的3日簡單平均；K反應較快，D較平滑。
    # 本版使用SMA平滑，與部分看盤軟體的遞迴平滑KD可能不同。
    # 80以上常稱高檔、20以下常稱低檔，但不代表即將反轉。
    k, d = data.indicator(
        "STOCH", adjust_price=True,
        fastk_period=5,
        slowk_period=3, slowk_matype=0,
        slowd_period=3, slowd_matype=0
    )
    k = k[str(stock_id)].sort_index()
    d = d[str(stock_id)].reindex(k.index)

    # 最近20個交易日的交叉需21筆完整資料，不跳過缺值。
    if len(k) < 21 or pd.concat([k, d], axis=1).iloc[-21:].isna().any().any():
        raise ValueError("最近21個交易日的KD資料不足，無法完整計算近期交叉")

    # 近3日變化：最新K－3個交易日前K，單位為指標點數。
    k_change_3d = k.iloc[-1] - k.iloc[-4]

    # 從最新一天往回計算K高於／低於D的連續交易日數。
    # 使用原始精度；相等或缺值就中斷，不額外限制為20日。
    gap = k - d
    above_d_days = 0
    below_d_days = 0
    for value in gap.iloc[::-1]:
        if pd.isna(value) or value <= 0:
            break
        above_d_days += 1
    for value in gap.iloc[::-1]:
        if pd.isna(value) or value >= 0:
            break
        below_d_days += 1

    # 合併向上／向下交叉，只保留最近一次交叉距今天數。
    # 前日相等、今日嚴格超越也算交叉；今日剛好相等不算。
    cross_up = (k.shift(1) <= d.shift(1)) & (k > d)
    cross_down = (k.shift(1) >= d.shift(1)) & (k < d)
    crossed = cross_up | cross_down
    cross_days_ago = -1
    for days_ago in range(20):
        if crossed.iloc[-1 - days_ago]:
            cross_days_ago = days_ago
            break

    # 輸出才四捨五入；連續天數可辨認四捨五入後看似相等的兩線關係。
    return {
        "k_5": round(float(k.iloc[-1]), 2),  # 最新K
        "d_5": round(float(d.iloc[-1]), 2),  # 最新D
        "k_change_3d": round(float(k_change_3d), 2),  # K近3日變化
        "kd_above_d_days": above_d_days,  # K連續高於D的交易日數
        "kd_below_d_days": below_d_days,  # K連續低於D的交易日數
        "kd_cross_days_ago_20d": cross_days_ago  # 0＝今天；1＝前一交易日；-1＝近20日未交叉
    }
kd_latest("2330")

{'k_5': 13.27,
 'd_5': 19.27,
 'k_change_3d': -39.65,
 'kd_above_d_days': 0,
 'kd_below_d_days': 3,
 'kd_cross_days_ago_20d': 2}

In [3]:
def macd_latest(stock_id):
    import pandas as pd

    # EMA＝指數移動平均，近期價格權重較高。
    # DIF＝6日EMA－13日EMA；正值表示6日EMA高於13日EMA。
    # signal（訊號線）＝DIF的5日EMA；hist（柱狀值）＝DIF－訊號線，不乘2。
    # DIF觀察均線關係，hist觀察DIF相對訊號線的位置；並非獨立買賣指令。
    dif, signal, hist = data.indicator(
        "MACD", adjust_price=True,
        fastperiod=6, slowperiod=13, signalperiod=5
    )
    dif = dif[str(stock_id)].sort_index()
    signal = signal[str(stock_id)].reindex(dif.index)
    hist = hist[str(stock_id)].reindex(dif.index)

    # 最近20日的交叉需要21筆完整資料，不跳過缺值。
    if len(dif) < 21 or pd.concat([dif, signal, hist], axis=1).iloc[-21:].isna().any().any():
        raise ValueError("最近21個交易日的MACD資料不足，無法完整計算近期交叉")

    # 總變化看方向；連續天數看變化是否延續，兩者搭配使用。
    hist_change_3d = hist.iloc[-1] - hist.iloc[-4]
    hist_change = hist.diff()
    hist_increasing_days = 0
    hist_decreasing_days = 0
    for change in hist_change.iloc[::-1]:
        if pd.isna(change) or change <= 0:
            break
        hist_increasing_days += 1
    for change in hist_change.iloc[::-1]:
        if pd.isna(change) or change >= 0:
            break
        hist_decreasing_days += 1

    # 合併向上／向下穿越訊號線，只記最近一次距今天數，不是DIF穿越零軸。
    # 前日相等、今日嚴格超越也算；今日剛好相等不算。
    cross_up = (dif.shift(1) <= signal.shift(1)) & (dif > signal)
    cross_down = (dif.shift(1) >= signal.shift(1)) & (dif < signal)
    crossed = cross_up | cross_down
    cross_days_ago = -1
    for days_ago in range(20):
        if crossed.iloc[-1 - days_ago]:
            cross_days_ago = days_ago
            break

    # hist負但增加＝負差距縮小；hist正但減少＝正差距縮小。
    # 先用原始精度計算再四捨五入；輸出0.00可能只是很接近0，不宜反推交叉方向。
    return {
        "macd_dif": round(float(dif.iloc[-1]), 2),  # 最新DIF
        "macd_hist": round(float(hist.iloc[-1]), 2),  # 最新柱狀值，不乘2
        "macd_hist_change_3d": round(float(hist_change_3d), 2),  # 最新柱狀值－3個交易日前柱狀值
        "macd_hist_increasing_days": hist_increasing_days,  # 連續增加次數，負值變得較不負也算增加
        "macd_hist_decreasing_days": hist_decreasing_days,  # 連續減少次數，持平或缺值中斷
        "macd_cross_days_ago_20d": cross_days_ago  # 0＝今天；1＝前一交易日；-1＝近20日未交叉
    }
macd_latest(2330)

Your version is 2.2.16, please install a newer version.
Use "pip install finlab==2.3.4" to update the latest version.


{'macd_dif': -104.96,
 'macd_hist': -25.84,
 'macd_hist_change_3d': -23.02,
 'macd_hist_increasing_days': 0,
 'macd_hist_decreasing_days': 4,
 'macd_cross_days_ago_20d': 3}

In [4]:
def bollinger_latest(stock_id):
    import pandas as pd

    # 布林通道：中軌為10日簡單平均，上下軌各距中軌1.9個標準差。
    # 重點：價格在通道哪裡、波動正在擴大或收斂、是否已持續一段時間。
    upper, middle, lower = data.indicator(
        "BBANDS", adjust_price=True,
        timeperiod=10, nbdevup=1.9, nbdevdn=1.9, matype=0
    )

    # 價格與軌道都使用還原價格，避免不同口徑造成位置判斷錯誤。
    price = data.get("etl:adj_close")[str(stock_id)].sort_index()
    upper = upper[str(stock_id)].reindex(price.index)
    middle = middle[str(stock_id)].reindex(price.index)
    lower = lower[str(stock_id)].reindex(price.index)

    # 標準化帶寬＝（上軌－下軌）／中軌×100%，中軌為0時無法計算。
    bandwidth = (upper - lower) / middle.where(middle != 0) * 100
    if len(bandwidth) < 60 or bandwidth.iloc[-60:].isna().any():
        raise ValueError("最近60個交易日的帶寬資料不足，無法計算60日百分位")
    if pd.isna(price.iloc[-1]):
        raise ValueError("最新交易日的還原收盤價缺漏，無法計算通道位置")

    # 價格位置：0＝下軌、50＝中軌、100＝上軌；保留通道外的值。
    width = upper.iloc[-1] - lower.iloc[-1]
    if width <= 0:
        raise ValueError("最新布林通道寬度須大於0，無法計算價格位置")
    position_pct = (price.iloc[-1] - lower.iloc[-1]) / width * 100

    # 近3日帶寬變化，單位為百分點；不是帶寬變化率。
    bandwidth_change_3d = bandwidth.iloc[-1] - bandwidth.iloc[-4]
    bandwidth_change = bandwidth.diff()

    # 從最新一天往回計算連續縮小／擴大的變化次數；持平或缺值就中斷。
    contraction_days = 0
    expansion_days = 0
    for change in bandwidth_change.iloc[::-1]:
        if pd.isna(change) or change >= 0:
            break
        contraction_days += 1
    for change in bandwidth_change.iloc[::-1]:
        if pd.isna(change) or change <= 0:
            break
        expansion_days += 1

    # 近60筆（含今天）中，帶寬嚴格小於今天的比例，範圍0～98.33。
    # 越低表示相對自己近60日越窄；相等不算較小，不代表即將向上突破。
    bandwidth_percentile_60d = float((bandwidth.iloc[-60:] < bandwidth.iloc[-1]).mean() * 100)

    # 原始值先計算，輸出才取兩位小數。位置與帶寬都不是獨立買賣訊號。
    return {
        "bollinger_position_pct": round(float(position_pct), 2),  # 價格在通道中的相對位置
        "bollinger_bandwidth_pct": round(float(bandwidth.iloc[-1]), 2),  # 最新標準化帶寬，%
        "bollinger_bandwidth_change_3d_pp": round(float(bandwidth_change_3d), 2),  # 近3日變化，百分點
        "bollinger_contraction_days": contraction_days,  # 連續收斂次數
        "bollinger_expansion_days": expansion_days,  # 連續擴張次數
        "bollinger_bandwidth_percentile_60d": round(bandwidth_percentile_60d, 2)  # 帶寬相對自身近60日的位置
    }
bollinger_latest(2330)

{'bollinger_position_pct': -3.79,
 'bollinger_bandwidth_pct': 8.67,
 'bollinger_bandwidth_change_3d_pp': 5.59,
 'bollinger_contraction_days': 0,
 'bollinger_expansion_days': 3,
 'bollinger_bandwidth_percentile_60d': 85.0}